# Entrenamiento LoRA (Low-Rank Adaptation) sobre Stable Diffusion 1.5
## Máster en Inteligencia Artificial, Cloud Computing y DevOps
## Módulo: Generacion de Imagenes con AI
## Autor: Alexander De Sousa

Este notebook presenta el proceso de entrenamiento mediante LoRA (Low-Rank Adaptation) aplicado a Stable Diffusion 1.5 para aprender el estilo visual de ilustraciones y portadas de libros antiguos.

A diferencia del finetuning completo, LoRA permite entrenar únicamente un conjunto reducido de adaptadores insertados en las capas de atención de la UNet, lo que reduce drásticamente:

- el tiempo de entrenamiento,
- el consumo de memoria,
- y el tamaño final del modelo.

El objetivo es obtener un modelo ligero y eficiente que capture el estilo del dataset, manteniendo la calidad visual y permitiendo su reutilización en cualquier pipeline compatible.
---


#### 1. Instalación de dependencias
Instala todas las librerías necesarias para entrenar y aplicar LoRA sobre Stable Diffusion.

In [1]:
pip install diffusers transformers accelerate peft safetensors datasets pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 13.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 16.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 44.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 47.6 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: fsspec0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/24 [pyarrow]
    Found existing installation: fsspec 2026.6.0━━━━━━━━━━━━━━  3/24 [pyarrow]
    Uninstalling fsspec-2026.6.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/24 [pyarrow]
      Successfully uninstalled fsspec-2026.6.0━━━━━━━━━━━━━━━━  3/24 [pyarrow]
  Attempting uninstall: huggingface-hub90m╺━━━━━━━━━━━━━━ 15/24 [multiprocess]]
    Found existing installation: huggingface-hub 0.23.4━━━━━━━ 15/24 [multiprocess]
    Uninstalling huggingface-hu

#### 2. Imports y configuración del dispositivo
Carga las clases principales.

In [6]:
import diffusers, peft
print("diffusers:", diffusers.__version__)
print("peft:", peft.__version__)

ImportError: cannot import name 'MIN_PEFT_VERSION' from 'diffusers.utils.constants' (/home/alexd/miniconda3/envs/lora-cpu/lib/python3.10/site-packages/diffusers/utils/constants.py)

In [3]:
import torch
from torch import nn
from diffusers import StableDiffusionPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from PIL import Image
import numpy as np


ImportError: cannot import name 'MIN_PEFT_VERSION' from 'diffusers.utils.constants' (/home/alexd/miniconda3/envs/lora-cpu/lib/python3.10/site-packages/diffusers/utils/constants.py)

#### 3. Cargar modelo base

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float32
).to("cpu")

unet = pipe.unet
text_encoder = pipe.text_encoder
tokenizer = pipe.tokenizer
noise_scheduler = pipe.scheduler

In [ ]:
def generate_before_after(pipe, unet_lora, prompt, out_prefix="result"):
    """
    Genera y guarda imágenes ANTES y DESPUÉS de aplicar LoRA moderno (PEFT).
    """

    # Imagen ANTES (modelo base)
    image_before = pipe(prompt).images[0]
    image_before.save(f"{out_prefix}_before.png")

    # Aplicar LoRA al UNet
    pipe.unet = unet_lora

    # Imagen DESPUÉS (modelo con LoRA)
    image_after = pipe(prompt).images[0]
    image_after.save(f"{out_prefix}_after.png")

    return image_before, image_after


#### 4. Cargar TU dataset real

In [ ]:
dataset = load_dataset("gigant/oldbookillustrations", split="train")

def preprocess(example):
    # Imagen principal
    image = example["1600px"].convert("RGB").resize((512, 512))

    # Texto descriptivo
    caption = example["info_alt"] if example["info_alt"] else "old book illustration"

    example["pixel_values"] = np.array(image).astype(np.float32) / 255.0
    example["prompt"] = caption
    return example

dataset = dataset.map(preprocess)

#### 5. Dataloader

In [ ]:
def collate_fn(batch):
    images = torch.tensor([b["pixel_values"] for b in batch]).permute(0, 3, 1, 2)
    prompts = [b["prompt"] for b in batch]
    tokens = tokenizer(prompts, padding="max_length", truncation=True, return_tensors="pt")
    return images, tokens.input_ids

from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

#### 6. Configurar LoRA moderno (PEFT)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.1,
    bias="none",
    task_type="UNET"
)

unet = get_peft_model(unet, lora_config)

#### 7. Entrenamiento

In [ ]:
optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)

for epoch in range(3):
    for step, (images, input_ids) in enumerate(dataloader):
        noise = torch.randn_like(images)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (images.shape[0],), dtype=torch.long)

        noisy_images = noise_scheduler.add_noise(images, noise, timesteps)
        encoder_hidden_states = text_encoder(input_ids)[0]

        model_pred = unet(noisy_images, timesteps, encoder_hidden_states).sample

        loss = nn.functional.mse_loss(model_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 50 == 0:
            print(f"Epoch {epoch} Step {step} Loss {loss.item()}")

#### 8. Guardar LoRA

In [ ]:
unet.save_pretrained("lora_output")

#### 9. Generar imágenes con tu LoRA

In [ ]:
pipe.unet = unet
image = pipe("engraved victorian illustration, old book style, vintage ink").images[0]
image

In [ ]:
prompt_test = "engraved victorian illustration, old book style, vintage ink"

before, after = generate_before_after(
    pipe,
    unet,               # UNet con LoRA aplicado
    prompt_test,
    out_prefix="oldbook_lora"
)

before, after
